# Build a state-year ELEC dataset

This notebook combines the annual EIA ELEC extracts into one row per U.S. state (including DC) and year. It produces:

- total plant net generation in megawatthours,
- average retail electricity prices by sector in cents per kilowatthour, and
- electricity customer accounts by sector.

The merge is an **outer join**, so years are retained even when one source does not cover them. National and multi-state regional series are excluded. The 2026 values in the source files may be partial and should not be treated as complete annual observations.

In [1]:
from pathlib import Path
import re

import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 1. Locate and load the annual ELEC files

The path logic works when the notebook is launched from either the repository root or the `Script` directory.

In [2]:
def find_project_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "Data" / "Electricity").is_dir():
            return candidate
    raise FileNotFoundError("Could not find Data/Electricity from the current directory.")

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "Data" / "Electricity"
OUTPUT_PATH = DATA_DIR / "ELEC.state_year.csv"

source_paths = {
    "generation": DATA_DIR / "ELEC.PLANT.GEN.A.csv",
    "price": DATA_DIR / "ELEC.PRICE.A.csv",
    "customers": DATA_DIR / "ELEC.CUSTOMERS.A.csv",
}

missing = [str(path) for path in source_paths.values() if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing source files: {missing}")

generation_raw = pd.read_csv(source_paths["generation"])
price_raw = pd.read_csv(source_paths["price"])
customers_raw = pd.read_csv(source_paths["customers"])

pd.DataFrame({
    "dataset": source_paths.keys(),
    "rows": [len(generation_raw), len(price_raw), len(customers_raw)],
    "columns": [generation_raw.shape[1], price_raw.shape[1], customers_raw.shape[1]],
})

,dataset,rows,columns
0,generation,57485,33
1,price,352,33
2,customers,289,26


## 2. Reshape annual series and keep state records

A valid state record has an ISO geography of the form `USA-XX`. This keeps all 50 states and the District of Columbia while removing national totals, Census-region aggregates, and generation series without a reported state.

In [3]:
META_COLS = ["series_id", "name", "geography", "iso3166", "sector", "units"]

def annual_to_long(df, value_name):
    df = df.drop(columns=[c for c in df.columns if str(c).lower().startswith("unnamed")], errors="ignore").copy()
    year_cols = [c for c in df.columns if re.fullmatch(r"\d{4}", str(c))]
    out = df.melt(
        id_vars=[c for c in META_COLS if c in df.columns],
        value_vars=year_cols,
        var_name="year",
        value_name=value_name,
    )
    out = out[out["iso3166"].astype("string").str.fullmatch(r"USA-[A-Z]{2}", na=False)].copy()
    out["state"] = out["iso3166"].str[-2:]
    out["year"] = pd.to_numeric(out["year"], errors="raise").astype("int64")
    out[value_name] = pd.to_numeric(out[value_name], errors="coerce")
    return out

# The plant file also contains fuel and prime-mover subtotals. Keep only the
# ALL-ALL series so each plant contributes exactly one total series.
generation_plant_totals = generation_raw[
    generation_raw["series_id"].astype("string").str.endswith("-ALL-ALL.A", na=False)
].copy()
generation_long = annual_to_long(generation_plant_totals, "generation_mwh")
price_long = annual_to_long(price_raw, "price_cents_per_kwh")
customers_long = annual_to_long(customers_raw, "customers")

pd.DataFrame({
    "dataset": ["generation", "price", "customers"],
    "state_records": [len(generation_long), len(price_long), len(customers_long)],
    "states": [generation_long.state.nunique(), price_long.state.nunique(), customers_long.state.nunique()],
    "first_year": [generation_long.year.min(), price_long.year.min(), customers_long.year.min()],
    "last_year": [generation_long.year.max(), price_long.year.max(), customers_long.year.max()],
})

,dataset,state_records,states,first_year,last_year
0,generation,402584,51,2001,2026
1,price,7462,51,2001,2026
2,customers,4465,51,2008,2026


## 3. Aggregate and pivot each measure

Generation is summed across the plant-level `ALL-ALL` series within each state-year. This filter is essential because the generation file also contains fuel and prime-mover subtotals that would otherwise double count generation. Prices and customer counts are already state-sector measures; they are pivoted into clearly named columns. Customer values can be fractional because the annual source values are annual averages of monthly account counts.

In [4]:
state_generation = (
    generation_long
    .groupby(["state", "year"], as_index=False)["generation_mwh"]
    .sum(min_count=1)
)

def sector_slug(sector):
    return re.sub(r"[^a-z0-9]+", "_", str(sector).strip().lower()).strip("_")

def pivot_sectors(df, value_col, prefix):
    duplicates = df.duplicated(["state", "year", "sector"], keep=False)
    if duplicates.any():
        examples = df.loc[duplicates, ["state", "year", "sector"]].head().to_dict("records")
        raise ValueError(f"Duplicate state-year-sector rows found: {examples}")
    wide = df.pivot(index=["state", "year"], columns="sector", values=value_col)
    wide.columns = [f"{prefix}_{sector_slug(col)}" for col in wide.columns]
    return wide.reset_index()

state_prices = pivot_sectors(price_long, "price_cents_per_kwh", "price_cents_per_kwh")
state_customers = pivot_sectors(customers_long, "customers", "customers")

state_generation.head()

,state,year,generation_mwh
0,AK,2001,"6,040,046.90"
1,AK,2002,"6,766,135.24"
2,AK,2003,"6,033,018.05"
3,AK,2004,"6,525,316.86"
4,AK,2005,"6,576,658.54"


## 4. Merge to one row per state-year

The outer joins preserve the union of years in all three sources. Missing values mean that the source did not report that measure for that state-year-sector.

In [5]:
state_year_elec = (
    state_generation
    .merge(state_prices, on=["state", "year"], how="outer", validate="one_to_one")
    .merge(state_customers, on=["state", "year"], how="outer", validate="one_to_one")
    .sort_values(["state", "year"])
    .reset_index(drop=True)
)

preferred_order = [
    "state", "year", "generation_mwh",
    "price_cents_per_kwh_all", "price_cents_per_kwh_residential",
    "price_cents_per_kwh_commercial", "price_cents_per_kwh_industrial",
    "price_cents_per_kwh_transportation", "price_cents_per_kwh_oth",
    "customers_all", "customers_residential", "customers_commercial",
    "customers_industrial", "customers_transportation",
]
state_year_elec = state_year_elec[[c for c in preferred_order if c in state_year_elec.columns]]

state_year_elec.head(10)

,state,year,generation_mwh,price_cents_per_kwh_all,price_cents_per_kwh_residential,price_cents_per_kwh_commercial,price_cents_per_kwh_industrial,price_cents_per_kwh_transportation,price_cents_per_kwh_oth,customers_all,customers_residential,customers_commercial,customers_industrial,customers_transportation
0,AK,2001,"6,040,046.90",10.54,12.12,10.29,7.61,NaN,14.37,NaN,NaN,NaN,NaN,NaN
1,AK,2002,"6,766,135.24",10.46,12.05,10.13,7.65,NaN,14.04,NaN,NaN,NaN,NaN,NaN
2,AK,2003,"6,033,018.05",10.50,11.98,10.49,7.86,NaN,0.00,NaN,NaN,NaN,NaN,NaN
3,AK,2004,"6,525,316.86",10.99,12.44,10.99,8.33,NaN,0.00,NaN,NaN,NaN,NaN,NaN
4,AK,2005,"6,576,658.54",11.72,13.30,11.56,9.29,NaN,0.00,NaN,NaN,NaN,NaN,NaN
5,AK,2006,"6,674,196.77",12.84,14.83,11.93,11.54,NaN,0.00,NaN,NaN,NaN,NaN,NaN
6,AK,2007,"6,821,391.62",13.28,15.18,12.19,12.63,NaN,0.00,NaN,NaN,NaN,NaN,NaN
7,AK,2008,"6,765,932.40",14.74,16.56,13.64,14.17,NaN,0.00,"317,244.17","268,824.08","47,068.92","1,351.17",NaN
8,AK,2009,"6,702,159.39",15.09,17.14,14.46,13.15,NaN,0.00,"318,458.75","269,745.58","47,347.67","1,365.50",NaN
9,AK,2010,"6,742,697.82",14.76,16.26,13.95,14.14,NaN,0.00,"321,149.67","271,955.42","47,843.83","1,350.42",NaN


## 5. Validate and export

The checks below enforce a unique state-year key, the expected 51 state/DC codes, and nonnegative reported measures before writing the CSV.

In [6]:
EXPECTED_STATES = {
    "AL", "AK", "AZ", "AR", "CA", "CO", "CT", "DE", "DC", "FL",
    "GA", "HI", "ID", "IL", "IN", "IA", "KS", "KY", "LA", "ME",
    "MD", "MA", "MI", "MN", "MS", "MO", "MT", "NE", "NV", "NH",
    "NJ", "NM", "NY", "NC", "ND", "OH", "OK", "OR", "PA", "RI",
    "SC", "SD", "TN", "TX", "UT", "VT", "VA", "WA", "WV", "WI", "WY",
}

assert not state_year_elec.duplicated(["state", "year"]).any(), "state-year key is not unique"
assert set(state_year_elec["state"]) == EXPECTED_STATES, "unexpected or missing state codes"
measure_cols = [c for c in state_year_elec.columns if c not in ["state", "year"]]
assert not (state_year_elec[measure_cols] < 0).any().any(), "negative measure found"

state_year_elec.to_csv(OUTPUT_PATH, index=False)

quality_summary = pd.Series({
    "output_path": str(OUTPUT_PATH),
    "rows": len(state_year_elec),
    "columns": state_year_elec.shape[1],
    "states": state_year_elec["state"].nunique(),
    "first_year": state_year_elec["year"].min(),
    "last_year": state_year_elec["year"].max(),
    "duplicate_state_year_keys": state_year_elec.duplicated(["state", "year"]).sum(),
    "missing_cells_pct": 100 * state_year_elec[measure_cols].isna().sum().sum() / state_year_elec[measure_cols].size,
})
quality_summary

output_path                  /Users/annagrace/Desktop/School/Drexel/Summer ...
rows                                                                      1326
columns                                                                     14
states                                                                      51
first_year                                                                2001
last_year                                                                 2026
duplicate_state_year_keys                                                    0
missing_cells_pct                                                        16.71
dtype: object

In [7]:
# Coverage by year makes partial or unavailable measures easy to spot.
coverage_by_year = (
    state_year_elec.groupby("year")[measure_cols]
    .count()
    .rename(columns=lambda c: f"{c}_states_reported")
)
coverage_by_year.tail(12)

,generation_mwh_states_reported,price_cents_per_kwh_all_states_reported,price_cents_per_kwh_residential_states_reported,price_cents_per_kwh_commercial_states_reported,price_cents_per_kwh_industrial_states_reported,price_cents_per_kwh_transportation_states_reported,price_cents_per_kwh_oth_states_reported,customers_all_states_reported,customers_residential_states_reported,customers_commercial_states_reported,customers_industrial_states_reported,customers_transportation_states_reported
year,,,,,,,,,,,,
2015,51,51,51,51,51,32,51,51,51,51,51,31
2016,51,51,51,51,51,32,51,51,51,51,51,31
2017,51,51,51,51,51,32,51,51,51,51,51,31
2018,51,51,51,51,51,32,51,51,51,51,51,31
2019,51,51,51,51,51,32,51,51,51,51,51,31
2020,51,51,51,51,51,32,51,51,51,51,51,31
2021,51,51,51,51,51,32,51,51,51,51,51,31
2022,51,51,51,51,51,32,51,51,51,51,51,31
2023,51,51,51,51,51,32,51,51,51,51,51,31
